# HAM10000 Pilot — Zero-Shot Skin Lesion Classification

**Model:** Qwen2.5-VL-3B-Instruct (zero-shot, no fine-tuning, no few-shot examples)
**Sample:** 105 images, stratified 15 per class across HAM10000's 7 diagnostic classes
**Purpose:** Small-scale pilot to demonstrate full analysis capability before the full 3-model study.

Run all cells top to bottom on a Colab or Kaggle GPU runtime (T4 or better).

In [3]:
!pip install -q transformers accelerate qwen-vl-utils pillow pandas scikit-learn torch

## Step 1 — Point this at your HAM10000 data

- **On Kaggle:** click *Add Data* → search "Skin Cancer MNIST: HAM10000" (by kmader) → add it.
  It mounts at `/kaggle/input/skin-cancer-mnist-ham10000/`.
- **On Colab:** upload your `kaggle.json`, then:
  ```
  !pip install -q kaggle
  !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
  !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p data --unzip
  ```
  then set `DATA_DIR = Path("data")` below.

In [4]:
import pandas as pd
from pathlib import Path

# EDIT THIS if you're on Colab or your mount path differs
DATA_DIR = Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000")

metadata = pd.read_csv(DATA_DIR / "HAM10000_metadata.csv")
print(metadata["dx"].value_counts())

dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


## Step 2 — Build the image path lookup and label names

In [5]:
import os

img_folders = [DATA_DIR / "HAM10000_images_part_1", DATA_DIR / "HAM10000_images_part_2"]
image_path_lookup = {}
for folder in img_folders:
    if folder.exists():
        for f in os.listdir(folder):
            if f.endswith(".jpg"):
                image_path_lookup[f.replace(".jpg", "")] = str(folder / f)

metadata["image_path"] = metadata["image_id"].map(image_path_lookup)
metadata = metadata.dropna(subset=["image_path"]).reset_index(drop=True)
print(f"Images found on disk: {len(metadata)}")

LABEL_NAMES = {
    "akiec": "Actinic keratoses / intraepithelial carcinoma",
    "bcc": "Basal cell carcinoma",
    "bkl": "Benign keratosis-like lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic nevi",
    "vasc": "Vascular lesions",
}

Images found on disk: 10015


## Step 3 — Stratified sample: 15 images per class (105 total)

In [6]:
SEED = 42
N_PER_CLASS = 15

sampled = (
    metadata.groupby("dx", group_keys=False)
    .apply(lambda g: g.sample(n=min(N_PER_CLASS, len(g)), random_state=SEED))
    .reset_index(drop=True)
)
print(f"Total sampled: {len(sampled)}")
sampled.to_csv("ham10000_pilot_sample.csv", index=False)
sampled[["image_id", "dx"]].head(10)

Total sampled: 105


/tmp/ipykernel_58/3799342364.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(N_PER_CLASS, len(g)), random_state=SEED))


,image_id,dx
0,ISIC_0024646,akiec
1,ISIC_0027753,akiec
2,ISIC_0032404,akiec
3,ISIC_0029041,akiec
4,ISIC_0030491,akiec
5,ISIC_0026626,akiec
6,ISIC_0026702,akiec
7,ISIC_0030730,akiec
8,ISIC_0032014,akiec
9,ISIC_0027303,akiec


## Step 4 — Load Qwen2.5-VL-3B-Instruct

In [7]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

...

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Step 5 — Zero-shot prompt and inference loop

The prompt lists all 7 categories and asks for just the category code, so scoring is a simple string match — no free-text parsing headaches.

In [8]:
from qwen_vl_utils import process_vision_info

CLASS_LIST_TEXT = "\n".join(f"- {code_}: {name}" for code_, name in LABEL_NAMES.items())

def build_prompt():
    return (
        "You are a dermatology assistant. Look at this dermatoscopic image and classify it "
        "into exactly ONE of the following categories. Respond with ONLY the category code "
        "(e.g. 'mel'), nothing else.\n\n"
        f"Categories:\n{CLASS_LIST_TEXT}"
    )

def predict(image_path):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": build_prompt()},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=32)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

def parse_label(raw_output):
    lowered = raw_output.lower()
    for code_ in LABEL_NAMES:
        if code_ in lowered:
            return code_
    return "unparsed"


In [9]:
results = []
for i, row in sampled.iterrows():
    raw = predict(row["image_path"])
    pred = parse_label(raw)
    results.append({
        "image_id": row["image_id"],
        "true_label": row["dx"],
        "predicted_label": pred,
        "raw_output": raw,
    })
    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{len(sampled)} done")

results_df = pd.DataFrame(results)
results_df.to_csv("ham10000_pilot_results.csv", index=False)
results_df.head()

10/105 done
20/105 done
30/105 done
40/105 done
50/105 done
60/105 done
70/105 done
80/105 done
90/105 done
100/105 done


,image_id,true_label,predicted_label,raw_output
0,ISIC_0024646,akiec,nv,nv
1,ISIC_0027753,akiec,mel,mel
2,ISIC_0032404,akiec,akiec,akiec
3,ISIC_0029041,akiec,bkl,bkl
4,ISIC_0030491,akiec,akiec,akiec


## Step 6 — Score the results

In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

valid = results_df[results_df["predicted_label"] != "unparsed"]
print(f"Parsed cleanly: {len(valid)}/{len(results_df)}")

acc = accuracy_score(valid["true_label"], valid["predicted_label"])
print(f"Overall accuracy: {acc:.3f}\n")

print("Per-class report:")
print(classification_report(valid["true_label"], valid["predicted_label"], zero_division=0))

labels = list(LABEL_NAMES.keys())
cm = confusion_matrix(valid["true_label"], valid["predicted_label"], labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
cm_df.to_csv("ham10000_pilot_confusion_matrix.csv")
cm_df

Parsed cleanly: 105/105
Overall accuracy: 0.229

Per-class report:
              precision    recall  f1-score   support

       akiec       0.50      0.20      0.29        15
         bcc       0.00      0.00      0.00        15
         bkl       0.15      0.27      0.20        15
          df       0.00      0.00      0.00        15
         mel       0.25      0.80      0.38        15
          nv       0.20      0.33      0.25        15
        vasc       0.00      0.00      0.00        15

    accuracy                           0.23       105
   macro avg       0.16      0.23      0.16       105
weighted avg       0.16      0.23      0.16       105



,pred_akiec,pred_bcc,pred_bkl,pred_df,pred_mel,pred_nv,pred_vasc
true_akiec,3,0,6,0,3,3,0
true_bcc,0,0,4,0,2,9,0
true_bkl,1,0,4,0,5,5,0
true_df,2,0,6,0,6,1,0
true_mel,0,0,1,0,12,2,0
true_nv,0,0,1,0,9,5,0
true_vasc,0,0,4,0,11,0,0


## Step 7 — Pull out the misclassified cases for your qualitative write-up

Look at a handful of these and write 1–2 sentences each on why the model likely got it wrong (e.g. visually similar classes, subtle lesion features, ambiguous coloring). This qualitative section is what Sir is really checking for.

In [11]:
wrong = results_df[results_df["predicted_label"] != results_df["true_label"]]
print(f"{len(wrong)} misclassified out of {len(results_df)}")
wrong[["image_id", "true_label", "predicted_label", "raw_output"]]

81 misclassified out of 105


,image_id,true_label,predicted_label,raw_output
0,ISIC_0024646,akiec,nv,nv
1,ISIC_0027753,akiec,mel,mel
3,ISIC_0029041,akiec,bkl,bkl
6,ISIC_0026702,akiec,bkl,bkl
7,ISIC_0030730,akiec,bkl,bkl
...,...,...,...,...
100,ISIC_0032538,vasc,bkl,bkl
101,ISIC_0029608,vasc,mel,mel
102,ISIC_0024475,vasc,mel,mel
103,ISIC_0026490,vasc,mel,mel
